In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if not candidate.suffix and (candidate.is_absolute() or ':' in text):
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [ ]:
import numpy as np
import pandas as pd
import json
from itertools import combinations
import itertools
import random
from collections import defaultdict
import os
import shutil

In [ ]:
# Read in the dataset of all complexes
all_complexes = pd.read_json("allComplexes.json")
all_complexes.head(5)

In [ ]:
# Read in the dataset of functional complex groups
fcg = pd.read_csv(DATA_DIR / 'fcg.txt', sep = "\t")
fcg.head(5)

In [ ]:
# Data pre-processing------
# All complexes' subunit genes
allGenes = all_complexes[all_complexes['Organism'] == 'Human']['subunits(Gene name)']
allGenes = allGenes.tolist()
allHumanComplexes = all_complexes[all_complexes['Organism'] == 'Human']

# only fcg
fcgGenes = fcg[fcg['Organism'] == 'Human']['subunits(Gene name)']
fcgGenes = fcgGenes.tolist()
fcgHumanComplexes = fcg[fcg['Organism'] == 'Human']
print(allGenes[:5])
print(fcgGenes[:5])

In [ ]:
# Positive Control List Generation------
def PositiveControlGeneration(df, geneList):
    
    eachComplex_dict = {}
    
    for i in range(len(df)):
        complexID = df.iloc[i]['ComplexID']
        genesInEachComplex = geneList[i].split(";")
        genesInEachComplex = list(set(genesInEachComplex))
        howManyGenes = len(genesInEachComplex)
        
        if howManyGenes > 2:
            random.shuffle(genesInEachComplex)
            genePairList = []
            
            if howManyGenes % 2 == 0:
                
                numGenePair = howManyGenes // 2

            else: 
                numGenePair = (howManyGenes - 1) // 2
            
                
            for j in range(numGenePair):
                genePairList.append([genesInEachComplex[2*j], genesInEachComplex[2*j + 1]])
            eachComplex_dict[complexID] = genePairList
                
        
        elif howManyGenes == 2:
            eachComplex_dict[complexID] = [genesInEachComplex]
        
        else:
            eachComplex_dict[complexID] = []
            
    return eachComplex_dict

        

In [ ]:
# Positive Control Lists Calling
FirstPositiveControl = PositiveControlGeneration(allHumanComplexes, allGenes)
SecondPositiveControl = PositiveControlGeneration(allHumanComplexes, allGenes)
ThirdPositiveControl = PositiveControlGeneration(allHumanComplexes, allGenes)
FourthPositiveControl = PositiveControlGeneration(allHumanComplexes, allGenes)
FifthPositiveControl = PositiveControlGeneration(allHumanComplexes, allGenes)

In [ ]:
# Create a matrix to who whether two distinct genes are in the same complexes------
# Create a list of strings of all unique genes------
rawGeneList = [string.split(';') for string in allGenes]
print(rawGeneList[:5])
flattenGeneList = list(itertools.chain.from_iterable(rawGeneList))
flattenGeneList = list(set(flattenGeneList))
print(flattenGeneList[:5])
len(flattenGeneList)

In [ ]:
# Universal truth matrix of positive interacting subunit-----
nrow = len(flattenGeneList)
ncol = len(flattenGeneList)

rows = [0 for number in range(nrow**2)]
values = np.array(rows)
matrix = values.reshape(-1, nrow)
matrix.shape == (nrow, ncol)
print(matrix.shape)
matrix

In [ ]:
# Create a dictionary that contains all unique human genes in one complex------
oneComplexGenes = dict()
for a in range(len(allHumanComplexes)):
    complex_id = int(allHumanComplexes.iloc[a]['ComplexID'])
    whatGenesinThatComplex = allHumanComplexes.iloc[a]['subunits(Gene name)']
    whatGenesinThatComplex = whatGenesinThatComplex.split(';')
    whatUniqueGenesinThatComplex = list(set(whatGenesinThatComplex))
    oneComplexGenes[complex_id] = whatUniqueGenesinThatComplex

In [ ]:
# Check wheter all unique human gene indice are included in the matrix-----
def flatten_dict_to_list(d):
    return [item for sublist in d.values() for item in sublist]

testComplexGenes = flatten_dict_to_list(oneComplexGenes)
len(list(set(testComplexGenes))) == nrow

In [ ]:
# Create a dictionary that summaries all associated genes with a gene
associatedGenes = dict()
for i in flattenGeneList:
    associatedList = []
    for j in oneComplexGenes.keys():
        if i in oneComplexGenes[j]:
            associatedGenesInThisComplex = [item for item in oneComplexGenes[j] if item != i]
            associatedList.extend(associatedGenesInThisComplex)
            associatedGenes[i] = list(set(associatedList))

In [ ]:
# Finally update the matrix If it is the same gene or they are in the same Complexes, then assign 1
for gene1_index in range(len(flattenGeneList)):
    for gene2_index in range(len(flattenGeneList)):
        gene1 = flattenGeneList[gene1_index]
        gene2 = flattenGeneList[gene2_index]
        if gene1 == gene2:
            matrix[gene1_index, gene2_index] = 1
        else:
            gene1associatedGenes = associatedGenes.get(gene1, [])

            if gene2 in gene1associatedGenes:
                matrix[gene1_index, gene2_index] = 1
            else:
                matrix[gene1_index, gene2_index] = 0

matrix # but this can only check whether they are in the same complex, not the final list for selecting negative control

In [ ]:
np.count_nonzero(matrix == 0)

In [ ]:
# Convert the Positive Control into a dataframe which is convenient for converting into a csv------
def flatten_gene_dict(gene_dict):
    listGenePairs = [gene_pair for sublist in gene_dict.values() for gene_pair in sublist]
    df = pd.DataFrame(listGenePairs, columns = ['Gene1', 'Gene2'])
    return df

FirstPositiveControlDF = flatten_gene_dict(FirstPositiveControl)
print(len(FirstPositiveControlDF))
FirstPositiveControlDF.head(10)

In [ ]:
SecondPositiveControlDF = flatten_gene_dict(SecondPositiveControl)
ThirdPositiveControlDF = flatten_gene_dict(ThirdPositiveControl)
FourthPositiveControlDF = flatten_gene_dict(FourthPositiveControl)
FifthPositiveControlDF = flatten_gene_dict(FifthPositiveControl)

In [ ]:
len(SecondPositiveControlDF)

In [ ]:
# Negative Control List Generation-------
def NegativeControlGeneration(pairedPositiveControlDF, interactingMatrix, flattenGenes):
    Gene1List = pairedPositiveControlDF['Gene1']
    Gene2List = pairedPositiveControlDF['Gene2'].tolist()
    random.shuffle(Gene2List)
    
    # Make a pseudo negative control df 
    pseudoNegPairDF = pd.DataFrame({'Gene1': Gene1List, 'Gene2': Gene2List})
    
    rows_to_drop = []
    for i in range(len(pseudoNegPairDF)):
        gene1 = pseudoNegPairDF.iloc[i]['Gene1']
        gene2 = pseudoNegPairDF.iloc[i]['Gene2']
        gene1_index = flattenGenes.index(gene1)
        gene2_index = flattenGenes.index(gene2)
        
        if interactingMatrix[gene1_index, gene2_index] != 0:
            rows_to_drop.append(i)
    
    pseudoNegPairDF = pseudoNegPairDF.drop(index = rows_to_drop)
            
    return pseudoNegPairDF

In [ ]:
FirstNegativeControl = NegativeControlGeneration(FirstPositiveControlDF, matrix, flattenGeneList)
FirstNegativeControl.head(5)

In [ ]:
len(FirstNegativeControl)

In [ ]:
SecondNegativeControl = NegativeControlGeneration(SecondPositiveControlDF, matrix, flattenGeneList)
ThirdNegativeControl = NegativeControlGeneration(ThirdPositiveControlDF, matrix, flattenGeneList)
FourthNegativeControl = NegativeControlGeneration(FourthPositiveControlDF, matrix, flattenGeneList)
FifthNegativeControl = NegativeControlGeneration(FifthPositiveControlDF, matrix, flattenGeneList)

In [ ]:
print(len(SecondNegativeControl), len(ThirdNegativeControl), len(FourthNegativeControl), len(FifthNegativeControl))

In [ ]:
# Sanity Check-------

def SanityCheck(NegativeControl):
    
    sameComplexCount = 0
    for i in range(len(NegativeControl)):
        gene1 = NegativeControl.iloc[i]['Gene1']
        gene2 = NegativeControl.iloc[i]['Gene2']
        gene1associatedGenes = associatedGenes[gene1]
        if gene2 in gene1associatedGenes:
            sameComplexCount += 1
            
    return sameComplexCount


In [ ]:
print(SanityCheck(FirstNegativeControl), SanityCheck(SecondNegativeControl), SanityCheck(ThirdNegativeControl), SanityCheck(FourthNegativeControl), SanityCheck(FifthNegativeControl))

In [ ]:
os.getcwd()

In [ ]:
FirstPositiveControlDF.to_csv(DATA_DIR / 'PPI_Pairs' / 'FirstPositiveControl.csv', index = False)
FirstNegativeControl.to_csv(DATA_DIR / 'PPI_Pairs' / 'FirstNegativeControl.csv', index = False)

In [ ]:
SecondPositiveControlDF.to_csv(DATA_DIR / 'PPI_Pairs' / 'SecondPositiveControl.csv', index = False)
SecondNegativeControl.to_csv(DATA_DIR / 'PPI_Pairs' / 'SecondNegativeControl.csv', index = False)

In [ ]:
ThirdPositiveControlDF.to_csv(DATA_DIR / 'PPI_Pairs' / 'ThirdPositiveControl.csv', index = False)
ThirdNegativeControl.to_csv(DATA_DIR / 'PPI_Pairs' / 'ThirdNegativeControl.csv', index = False)

In [ ]:
FourthPositiveControlDF.to_csv(DATA_DIR / 'PPI_Pairs' / 'FourthPositiveControl.csv', index = False)
FourthNegativeControl.to_csv(DATA_DIR / 'PPI_Pairs' / 'FourthNegativeControl.csv', index = False)

In [ ]:
FifthPositiveControlDF.to_csv(DATA_DIR / 'PPI_Pairs' / 'FifthPositiveControl.csv', index = False)
FifthNegativeControl.to_csv(DATA_DIR / 'PPI_Pairs' / 'FifthNegativeControl.csv', index = False)

In [ ]:
folder_name = 'AllPositiveAndNegativeControl'
os.makedirs(folder_name, exist_ok = True)
files_to_move = [f for f in os.listdir() if f.endswith('.csv')]

for file in files_to_move:
    shutil.move(file, os.path.join(folder_name, file))